# Imports

In [ ]:
import pandas as pd
import pyrolite.geochem
from pyrolite.geochem.ind import get_cations
from pyrolite.geochem.transform import oxide_conversion

pd.set_option('display.max_columns', None)

# Inputs

In [2]:
# input path to original dataset
input_path = r'../data/mnz_compiled.csv'

# output path for cleaned dataset
output_path = r'../data/mnz_cleaned.csv'

# Pre-processing

In [33]:
##### pre-processing & cleaning...
df = pd.read_csv(input_path)
df.rename(columns={
    'Age Ma': 'age', 
    'Error 1sig': 'error'
    }, 
    inplace=True)
oxide_cols_start = df.columns.get_loc('CaO')
oxide_cols = df.columns[oxide_cols_start:]
df[oxide_cols] = df[oxide_cols].apply(pd.to_numeric, errors='coerce')


##### drop analyses >5% analytical error...
loc = df.columns.get_loc('error') + 1
value = df['error'] / df['age'] * 100
df.insert(loc=loc, column='error_percent', value=value)
mask = df['error_percent'] <= 5
df = df.loc[mask].copy()


##### clean oxides...
loc = df.columns.get_loc('CaO')
value = df[oxide_cols].sum(axis=1, min_count=1)
df.insert(loc=loc, column='oxide_total', value=value)
df[oxide_cols] = df[oxide_cols].clip(lower=0, upper=None)


##### convert oxide wt% to element wt%...
elements = [str(get_cations(oxide)[0]) for oxide in oxide_cols]     # identify elements from oxides
for oxide, element in zip(oxide_cols, elements):
    df[element] = oxide_conversion(oxide, element)(df[oxide])


##### add LREEs & HREEs...
lree_cols = ['La', 'Ce', 'Pr', 'Nd', 'Sm', 'Eu']
hree_cols = ['Gd', 'Tb', 'Dy', 'Ho', 'Er', 'Tm', 'Yb']
df['LREE'] = df[lree_cols].sum(axis=1, min_count=1)
df['HREE'] = df[hree_cols].sum(axis=1, min_count=1)


##### indicators for CMZI garnet and cordierite lithologies...
mask_grt = (df['sample'] == 'WA15-4') | (df['sample'] == 'WA15-5')
mask_crd = (df['sample'] == 'WA15-4') | (df['sample'] == 'WA15-5') | (df['sample'] == 'WA14-002')
df['garnet'] = (~mask_grt).astype(int)
df['cordierite'] = mask_crd.astype(int)

##### define analysis zones...
# NOTE: same as zone, but CMZI split into garnet-bearing & garnet-absent (cordierite-bearing)
df['analysis_zone'] = df['zone']
df.loc[df['analysis_zone'] == 'CMZI', 'analysis_zone'] = 'CMZI (Grt)'
cmzi_mask = (df['zone'] == 'CMZI') & (df['garnet'] == 0) & (df['cordierite'] == 1)
df.loc[cmzi_mask, 'analysis_zone'] = 'CMZI (Crd)'


##### save cleaned dataset...
df.sort_values(['analysis_zone', 'age'], inplace=True)
df.reset_index(inplace=True, drop=True)
df.to_csv(output_path, index=False)


df.head()

,date,sample,lithology,zone,mnz,mnz_description,mnz_xraymap,domain,domain_reclassified,n,U WT%,Th WT%,Pb WT%,age,error,error_percent,oxide_total,CaO,K2O,SiO2,SrO,P2O5,As2O5,ThO2,UO2,Y2O3,La2O3,Ce2O3,Pr2O3,Nd2O3,Sm2O3,Eu2O3,Gd2O3,Tb2O3,Dy2O3,Ho2O3,Er2O3,Tm2O3,Yb2O3,SO3,PbO,Ca,K,Si,Sr,P,As,Th,U,Y,La,Ce,Pr,Nd,Sm,Eu,Gd,Tb,Dy,Ho,Er,Tm,Yb,S,Pb,LREE,HREE,garnet,cordierite,analysis_zone
0,10/6/2015,219-3-L,Rangeley,CBSZ,m41,rim,Y enriched overgrowth,rim,overgrowth,5.0,0.340177,4.293720,0.075651,313.460000,5.481569,1.748730,100.072302,0.939894,0.005426,0.178864,0.016206,30.491480,0.000000,4.885856,0.385909,1.378837,15.573740,29.674400,2.874736,10.985060,0.984231,0.225811,1.004706,0.041115,0.356465,0.012295,0.087689,0.040820,0.013642,0.007318,0.081493,0.671738,0.004504,0.083608,0.013703,13.307300,0.000000,4.293748,0.340179,1.085757,13.279466,25.335106,2.456381,9.418107,0.848762,0.195014,0.871676,0.035721,0.310595,0.010733,0.076686,0.035742,0.01198,0.002931,0.075652,51.532835,1.353134,1,0,CBSZ
1,12/11/2015,P14-005A-5B,Rangeley,CBSZ,m19,D4 top rim,Th enriched rim,D4,rim,5.0,0.271739,3.868724,0.066658,314.520000,9.312110,2.960737,100.285231,0.872301,0.018543,0.293431,0.000000,30.553220,0.000000,4.402254,0.308271,0.542359,14.367420,28.824320,3.087780,12.585000,2.143510,0.204734,1.707824,0.053391,0.346062,0.000000,0.000000,0.000096,0.000000,0.016773,0.071806,0.623430,0.015393,0.137160,0.000000,13.334245,0.000000,3.868753,0.271741,0.427077,12.250857,24.609333,2.638421,10.789825,1.848480,0.176811,1.481697,0.046387,0.301531,0.000000,0.000000,0.000084,0.00000,0.006717,0.066659,52.313727,1.829699,1,0,CBSZ
2,10/6/2015,9-1-L,Rangeley,CBSZ,m24,top rim,Y enriched rim,rim,rim,6.0,0.494592,3.986915,0.080217,321.750000,3.253793,1.011280,100.544990,0.974370,0.003835,0.230996,0.015805,30.308217,0.051551,4.536747,0.561084,0.681071,14.758250,29.148267,3.008727,12.078033,1.527837,0.227522,1.818587,0.077031,0.364254,0.000000,0.035816,0.065713,0.000000,0.009389,0.086412,0.696378,0.003183,0.107976,0.013365,13.227319,0.033609,3.986947,0.494596,0.536305,12.584111,24.885909,2.570872,10.355174,1.317547,0.196492,1.577794,0.066925,0.317382,0.000000,0.031321,0.057539,0.00000,0.003760,0.080218,51.910105,2.050961,1,0,CBSZ
3,10/6/2015,9-1-L,Rangeley,CBSZ,m18,top rim,Y enriched rim,rim,rim,6.0,0.514405,4.247175,0.085232,322.966667,2.000444,0.619397,100.286011,1.024968,0.012410,0.104117,0.000000,30.477800,0.007096,4.832897,0.583560,0.973683,14.580233,28.513217,2.888662,11.831450,1.485123,0.234208,1.960742,0.117715,0.492718,0.000000,0.047651,0.071852,0.000000,0.011011,0.091814,0.732541,0.010302,0.048668,0.000000,13.301329,0.004626,4.247207,0.514408,0.766721,12.432319,24.343723,2.468280,10.143764,1.280713,0.202266,1.701127,0.102272,0.429315,0.000000,0.041672,0.062914,0.00000,0.004410,0.085233,50.871065,2.337299,1,0,CBSZ
4,10/6/2015,9-1-L,Rangeley,CBSZ,m2,top rim,Y enriched rim,rim,rim,6.0,0.329983,3.909598,0.071973,323.733333,3.685889,1.138557,100.505853,0.937878,0.042255,0.109674,0.015325,30.418650,0.000000,4.448767,0.374344,1.054142,14.300350,29.031883,3.058748,12.350767,1.534922,0.194783,2.069160,0.144505,0.555123,0.000000,0.038243,0.038488,0.000000,0.006797,0.077532,0.670297,0.035078,0.051266,0.012959,13.275515,0.000000,3.909629,0.329985,0.830077,12.193667,24.786545,2.613614,10.589004,1.323657,0.168218,1.795190,0.125547,0.483690,0.000000,0.033444,0.033700,0.00000,0.002722,0.071974,51.674705,2.471571,1,0,CBSZ
